# fst2framegraph Colab runner

Run each cell in order. Upload the project zip when prompted if the notebook is not already inside an extracted project directory, then upload the OxCCAL CSV when prompted.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys
import zipfile
from google.colab import drive

os.environ.setdefault('USE_TF', '0')
os.environ.setdefault('TRANSFORMERS_NO_TF', '1')
os.environ.setdefault('USE_FLAX', '0')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

root = Path.cwd()
if not (root / 'pyproject.toml').exists():
    from google.colab import files
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.endswith('.zip')]
    if not zip_names:
        raise FileNotFoundError('Upload the fst2framegraph project zip.')
    project_dir = Path('/content/fst2framegraph')
    if project_dir.exists():
        shutil.rmtree(project_dir)
    project_dir.mkdir(parents=True)
    with zipfile.ZipFile(zip_names[0]) as zf:
        zf.extractall(project_dir)
    candidates = [p for p in project_dir.rglob('pyproject.toml')]
    if not candidates:
        raise FileNotFoundError('The uploaded zip does not contain pyproject.toml.')
    root = candidates[0].parent
    os.chdir(root)

wheel = root / 'wheels' / 'sentencepiece-0.2.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl'
if not wheel.exists():
    subprocess.check_call([sys.executable, 'scripts/fetch_wheels.py'])

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--force-reinstall', str(wheel)])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--find-links=wheels/', '-e', '.'])

try:
    import frame_semantic_transformer  # noqa: F401
except Exception:
    subprocess.check_call([sys.executable, 'scripts/install_colab_fst.py'])

framebase_dir = root / 'data' / 'framebase'
framebase_core = framebase_dir / 'FrameBase_schema_core.ttl.gz'
framebase_index = framebase_dir / 'framebase_index.sqlite'
if not framebase_core.exists() and not framebase_index.exists():
    subprocess.check_call(['fst2framegraph', 'setup-framebase', '--out', str(framebase_dir)])
if not framebase_core.exists() and not framebase_index.exists():
    raise FileNotFoundError('FrameBase setup did not produce a schema or index.')

drive.mount("/content/drive")
output_root = Path("/content/drive/MyDrive/fst2framegraph_outputs")
output_root.mkdir(parents=True, exist_ok=True)

print(f'Google Drive output root: {output_root}')
print(f'Project ready at {Path.cwd()}')

In [ ]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()
csv_names = [name for name in uploaded if name.lower().endswith('.csv')]
if not csv_names:
    raise FileNotFoundError('Upload an OxCCAL-style CSV file.')
csv_path = Path(csv_names[0])
csv_path

In [ ]:
import pandas as pd
from run_pipeline import run_pipeline

result = run_pipeline(
    csv_path,
    output_root=output_root,
    framebase_dir='data/framebase',
    require_real_fst=True,
)
display(pd.DataFrame([{
    'rows_in': result['rows_in'],
    'prepared_rows': result['rows_prepared'],
    'skipped_empty': result['rows_skipped_empty'],
    'documents': result['documents'],
    'graph_nodes': result['graph_nodes'],
    'graph_edges': result['graph_edges'],
    'lift_rows': result['lift_rows'],
    'reified_edges': result['reified_edges'],
    'dereified_edges': result['dereified_edges'],
    'validated_frames': result['framebase_validated_frames'],
    'validated_frame_elements': result['framebase_validated_frame_elements'],
}]))

lift = pd.read_csv(result['lift_path'])
display(lift)

frames = pd.read_csv(Path(result['framebase_reified_dir']) / 'frame_instances.csv')
elements = pd.read_csv(Path(result['framebase_reified_dir']) / 'frame_elements.csv')
direct_edges = pd.read_csv(Path(result['framebase_reified_dir']) / 'direct_edges.csv')
display(frames.head(20))
display(elements.head(20))
display(direct_edges.head(20))

print(result['summary_path'])
print(result['graph_path'])
print(result['framebase_reified_dir'])
print(f'Saved under Google Drive folder: {output_root}')